In [ ]:
!pip install vaderSentiment
import vaderSentiment
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from google.colab import drive
drive.mount('/content/drive')

     |████████████████████████████████| 133kB 16.0MB/s 
Mounted at /content/drive


In [ ]:
reddit_data_folder = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/Reddit Posts/'
file_name = 'afghan hound.json'
breed_reddit_json = json.load(open(reddit_data_folder + file_name))

In [ ]:
analyzer = SentimentIntensityAnalyzer()
for sentence in breed_reddit_json:
    vs = analyzer.polarity_scores(sentence)
    print("{:-<65} {}".format(sentence, str(vs)))

Word2Vec Analysis

In [ ]:
# All Import Statements Defined Here
# Note: Do not add to this list.
# All the dependencies you need, can be installed by running .
# ----------------
 
import sys
assert sys.version_info[0]==3
assert sys.version_info[1] >= 5
 
from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]
import nltk
nltk.download('reuters')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import reuters
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA
 
START_TOKEN = '<START>'
END_TOKEN = '<END>'
 
np.random.seed(0)
random.seed(0)
# ----------------

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [ ]:
def load_word2vec():
    """ Load Word2Vec Vectors
        Return:
            wv_from_bin: All 3 million embeddings, each lengh 300
    """
    import gensim.downloader as api
    wv_from_bin = api.load("word2vec-google-news-300")
    # wv_from_bin = api.load("glove-wiki-gigaword-200")
    vocab = list(wv_from_bin.vocab.keys())
    print("Loaded vocab size %i" % len(vocab))
    return wv_from_bin
  
# -----------------------------------
# Run Cell to Load Word Vectors
# Note: This may take several minutes
# -----------------------------------
wv_from_bin = load_word2vec()

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Loaded vocab size 3000000


Extracting all adjectives

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import json
dog_breeds_file = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/StandardDogBreeds.json'
try:
    dog_breeds_json = json.load(open(dog_breeds_file))
except FileNotFoundError:
    print('WARNING: Database file not found.')


list_of_subbreeds = [(subbreed + ' ' + breed) for breed, list_of_subbreeds in dog_breeds_json.items() for subbreed in list_of_subbreeds ]
list_of_breeds = [breed for breed, list_of_subbreeds in dog_breeds_json.items()]
full_list_of_dogs = list_of_subbreeds + list_of_breeds

In [ ]:
# full_list_of_dogs
dog_name = 'eskimo'
reddit_data_folder = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/Reddit Posts/'
file_name = dog_name + '.json'
dog_comments_json = json.load(open(reddit_data_folder + file_name))

In [ ]:
# ### Get list of adjectives from text
def adjectives_from_sentence(sentence):
  tokenized_sentence = nltk.word_tokenize(sentence)
  tagged_sentence = nltk.pos_tag(tokenized_sentence)
  set_of_adjectives = set()
  for word, tag in tagged_sentence:
    if tag == 'JJ':
      set_of_adjectives.add(word)
  return set_of_adjectives
 
set_of_adjectives = set()
reddit_data_folder = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/Reddit Posts/'
 
for dog_name in ['labrador', 'pitbull']:
  file_name = dog_name + '.json'
  dog_comments_json = json.load(open(reddit_data_folder + file_name))
  for sentence in dog_comments_json:
    set_of_adjectives = set_of_adjectives | adjectives_from_sentence(sentence)

In [ ]:
set_of_adjectives

{'this.I',
 'well-socialized…it',
 'petulant',
 'peanut',
 'ohh',
 'bite-force',
 '//www.equalityhumanrights.com/en/publication-download/take-lead-guide-welcoming-customers-assistance-dogs',
 'profound',
 'prospect',
 'coming',
 'Small',
 'stuffed',
 'teal',
 'inferior',
 'damm',
 'considérée',
 '//www.dyson.co.uk/support/journey/tools/921001-01',
 'formaient',
 'person-',
 'envahis',
 'type-',
 'social',
 'homeless',
 'unsourced',
 'surprising',
 'essential',
 'unicorn',
 'funny',
 'vous',
 'encontra-me',
 'insured',
 'pet-grade',
 'is.I',
 'boy',
 'geographical',
 'alien',
 "n'importe",
 'tasks-',
 'to/tear',
 'esa',
 'novo',
 'whiter',
 'dual',
 'Severe',
 '/',
 'changent',
 'second-class',
 'misanthropic',
 'asian',
 'spend',
 'booped',
 'lenient',
 'close',
 'dipshit',
 'toy',
 'nutrish',
 'graphic',
 'pointless',
 'spinal',
 'várias',
 'available',
 'frustrated',
 'competitive',
 'threaten',
 'DNA',
 'onde',
 'butt-hurt',
 'déduis',
 'Mexican',
 'veulent',
 'pepper',
 'postish',


In [ ]:
import math
word = 'good'
goodness_unit_vector = (wv_from_bin.word_vec('good') - wv_from_bin.word_vec('bad')) / np.linalg.norm((wv_from_bin.word_vec('good') - wv_from_bin.word_vec('bad')))
import numpy as np

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

# The more positive the value is the more the vector aligns with the goodness vector, if it is negative, it is opposite to good
def cosine(word, goodness_unit_vector):
  cosine = math.cos(angle_between(wv_from_bin.word_vec(word), goodness_unit_vector))
  return cosine
  
### Reduce the list of adjectives that most align and misalign with the goodness vector
adj_goodness_alignment = {}
for adj in set_of_adjectives:
  try:
    adj_goodness_alignment[adj] = cosine(adj, goodness_unit_vector)
  except:
    pass

sorted_adj_goodness_alignment = sorted(adj_goodness_alignment.items(), key=lambda item: item[1])
NUMBER_OF_TOP_WORDS = 100
positive_reddit_adj_with_positivity_scores = sorted_adj_goodness_alignment[-1 * NUMBER_OF_TOP_WORDS:]
negative_reddit_adj_with_positivity_scores = sorted_adj_goodness_alignment[:NUMBER_OF_TOP_WORDS]
positive_reddit_adj = [word for word, score in positive_reddit_adj_with_positivity_scores]
negative_reddit_adj = [word for word, score in negative_reddit_adj_with_positivity_scores]
all_top_reddit_adj = positive_reddit_adj + negative_reddit_adj

In [ ]:
sorted_adj_goodness_alignment[-200:]
# all_top_reddit_adj

In [ ]:
## Create Table: Adjective, Dog Breed, cosine_score
## NOTE: I say cosine score but I really did cosine angle, which is the opposite of cosine score
import pandas as pd
data = {}
data['adj'] = []
data['breed'] = []
data['cosine_scores'] = []
for adj in all_top_reddit_adj:
  for breed in list_of_breeds:
    try:
      # cosine_score = wv_from_bin.distance(adj, breed)
      if breed == "pitbull" and adj == "vicious":
        print("Pitbull viciousness", math.cos(angle_between(wv_from_bin.word_vec(adj), wv_from_bin.word_vec(breed))))
      cosine_score = math.cos(angle_between(wv_from_bin.word_vec(adj), wv_from_bin.word_vec(breed)))
      data['adj'].append(adj)
      data['breed'].append(breed)
      data['cosine_scores'].append(cosine_score)
    except KeyError:
      pass

adjective_scores_table = pd.DataFrame.from_dict(data)

Pitbull viciousness 0.41776150465011586


In [ ]:
adjective_scores_table
# adjective_scores_table.to_csv("AdjectiveScoreForEachAdjectiveBreedPair.csv")

,adj,breed,cosine_scores
0,full,affenpinscher,-0.020013
1,full,akita,0.048769
2,full,australian,0.067984
3,full,basenji,-0.080270
4,full,beagle,0.006382
...,...,...,...
12795,careless,terrier,0.081605
12796,careless,vizsla,0.093382
12797,careless,weimaraner,0.085138
12798,careless,whippet,0.081167


In [ ]:
# The below is wrong because it would give a positive score if aligns well with negative word also
# adjective_scores_table['cosine_scores'].filter(lambda value: value < 0)
# adjective_scores_table['is_positive'] = adjective_scores_table.apply(lambda x: True if x['cosine_scores'] > 0 else False, axis=1) 
# adjective_scores_table.groupby(['is_positive']).count()

In [ ]:
def get_top_n_adjectives_for(dog_breed, n=5, adjective_scores_table=adjective_scores_table, get_least_instead_of_top=False):
  adj_groupby_breed = adjective_scores_table.groupby(["breed"])
  if get_least_instead_of_top:
    return adjective_scores_table.iloc[adj_groupby_breed["cosine_scores"].nsmallest(n)[dog_breed].index]
  return adjective_scores_table.iloc[adj_groupby_breed["cosine_scores"].nlargest(n)[dog_breed].index]
 
get_top_n_adjectives_for("pitbull", n=10)
get_top_n_adjectives_for("labrador", n=10)
 
def get_top_n_breeds_that_are(adjective, n=5, adjective_scores_table=adjective_scores_table, get_least_instead_of_top=False):
  data = {}
  data['adj'] = []
  data['breed'] = []
  data['cosine_scores'] = []
  for breed in list_of_breeds:
    try:
      # cosine_score = wv_from_bin.distance(adj, breed)
      cosine_score = math.cos(angle_between(wv_from_bin.word_vec(adjective), wv_from_bin.word_vec(breed)))
      if breed == "pitbull" and adjective=="vicious":
        print("Pitbull viciousness:", math.cos(angle_between(wv_from_bin.word_vec(adjective), wv_from_bin.word_vec(breed))))
      data['adj'].append(adjective)
      data['breed'].append(breed)
      data['cosine_scores'].append(cosine_score)
    except KeyError:
      pass
 
  adjective_scores_table = pd.DataFrame.from_dict(data)
  adj_groupby_breed = adjective_scores_table.groupby(["adj"])
  if get_least_instead_of_top:
    return adjective_scores_table.iloc[adj_groupby_breed["cosine_scores"].nsmallest(n)[adjective].index]
  return adjective_scores_table.iloc[adj_groupby_breed["cosine_scores"].nlargest(n)[adjective].index]
 
get_top_n_breeds_that_are("affordable", n=1)
get_top_n_breeds_that_are("vicious", n=10)
 
def get_adjectives_that_this_breed_outperforms_in(breed, adjective_scores_table=adjective_scores_table):
  ### This fucntion is different from get top breeds in the sense that there might be adjectives that might not necessarily
  ### be the top adjective for this specific dog, but then again, these adjectives are used only for this dog and no other dog.
  ### For example, the top vicious might be the top adjective for a pitbull, and reckless might be the 7th top adjective, but
  ### knowing that pitbulls are seen as reckless is important because other dogs are described as reckless, so is still a adjective that needs study
  
  # Display the breeds with the max cosine score for each adj
  most_align = adjective_scores_table.groupby(['breed']).max().reset_index()[['breed', 'cosine_scores']]
  least_align = adjective_scores_table.groupby(['breed']).min().reset_index()[['breed', 'cosine_scores']]
 
  # for adj in most_align[['adj', 'cosine_scores']]:
 
  # Which breed most represents an adjective
  align_with_breed = {}
  align_with_breed['breed'] = []
  align_with_breed['adj most align'] = []
  align_with_breed['cosine most align'] = []
  align_with_breed['adj least align'] = []
  align_with_breed['cosine least align'] = []
 
  ## Extracting the breed that corresponds to the max cosine score found from groupby
  for breed, max_cosine_score, min_cosine_score in zip(most_align['breed'], most_align['cosine_scores'], least_align['cosine_scores']):
    filter = adjective_scores_table.loc[adjective_scores_table['breed'] == breed]
    filter_max =  filter.loc[filter['cosine_scores'] == max_cosine_score]
    align_with_breed['breed'].append(breed)
    align_with_breed['adj most align'].append(filter_max['adj'].values[0])
    align_with_breed['cosine most align'].append(max_cosine_score)
    filter_min  =  filter.loc[filter['cosine_scores'] == min_cosine_score]
    align_with_breed['adj least align'].append(filter_min['adj'].values[0])
    align_with_breed['cosine least align'].append(min_cosine_score)
 
  align_with_breed = pd.DataFrame.from_dict(align_with_breed)
  print(align_with_breed)
  # return align_with_adj.loc[align_with_adj['breed most align'] == 'pitbull']
 
get_adjectives_that_this_breed_outperforms_in("pitbull", adjective_scores_table=adjective_scores_table)

Pitbull viciousness: 0.41776150465011586
            breed adj most align  ...  adj least align cosine least align
0   affenpinscher        Hideous  ...       productive          -0.153087
1           akita       deformed  ...       meaningful          -0.148529
2      australian          dodgy  ...          cordial          -0.082133
3         basenji       sociable  ...       meaningful          -0.085070
4          beagle         widdle  ...         Negative          -0.061651
..            ...            ...  ...              ...                ...
59        terrier        vicious  ...            ample          -0.065694
60         vizsla         widdle  ...        equitable          -0.086334
61     weimaraner       sociable  ...       meaningful          -0.094788
62        whippet         classy  ...         damaging          -0.083452
63      wolfhound         widdle  ...        important          -0.084980

[64 rows x 5 columns]


# API

## Function Use Cases:


1.   **get_top_n_adjectives_for**(dog_breed, n=5 adjective_scores_table=adjective_scores_table, get_least_instead_of_top=False): Used to find the top adjectives describing a certain dog. 

For example, 

*get_top_n_adjectives_for("pitbull", 5, adjective_scores_table)*

*get_top_n_adjectives_for("labrador", n=1)*


2.   **get_top_n_breeds_that_are**(adjective, n=5, adjective_scores_table=adjective_scores_table, get_least_instead_of_top=False):
Find the top breed's that embody a certain concept (adjective)

For example, 
*get_top_n_breeds_that_are("affordable", n=1)*

*get_top_n_breeds_that_are("vicious", n=10)*


3.   **get_adjectives_that_this_breed_outperforms_in**("pitbull", adjective_scores_table=adjective_scores_table): Returns all the adjectives/concepts that a specified breed tops the charts in. This fucntion is different from get top breeds in the sense that there might be adjectives that might not necessarily be the top adjective for this specific dog, but then again, these adjectives are used only for this dog and no other dog. For example, the top vicious might be the top adjective for a pitbull, and reckless might be the 7th top adjective, but knowing that pitbulls are seen as reckless is important because other dogs are described as reckless, so is still a adjective that needs study.

For example,
*get_adjectives_that_this_breed_outperforms_in("pitbull", adjective_scores_table=adjective_scores_table)*

# Sample Runs of API

In [ ]:
print("Top 5 adjectives that describe a pitbull")
get_top_n_adjectives_for("pitbull", 5, adjective_scores_table)

Top 5 adjectives that describe a pitbull


,adj,breed,cosine_scores
9001,vicious,pitbull,0.417762
7977,heartless,pitbull,0.301660
9321,dumb,pitbull,0.271701
7913,stupid,pitbull,0.259636
12137,victim,pitbull,0.258613


In [ ]:
print("Top adjective that describes a labrador")
get_top_n_adjectives_for("labrador", 1)

Top adjective that describes a labrador


,adj,breed,cosine_scores
223,sociable,labrador,0.285398


In [ ]:
print("Most affordable breed according to redditors")
get_top_n_breeds_that_are("affordable", n=1)

Most affordable breed according to redditors


,adj,breed,cosine_scores
37,affordable,mix,0.11047


In [ ]:
print("Most vicious breed according to redditors")
get_top_n_breeds_that_are("vicious", n=10)

Most vicious breed according to redditors
Pitbull viciousness: 0.41776150465011586


,adj,breed,cosine_scores
41,vicious,pitbull,0.417762
50,vicious,rottweiler,0.388305
8,vicious,bulldog,0.353754
49,vicious,ridgeback,0.325882
27,vicious,hound,0.320133
3,vicious,basenji,0.294675
21,vicious,doberman,0.289458
43,vicious,pomeranian,0.286238
63,vicious,wolfhound,0.285203
45,vicious,pug,0.268567


In [ ]:
print("List of adjectives that pitbulls most outperform other dogs in")
get_adjectives_that_this_breed_outperforms_in("pitbull", adjective_scores_table=adjective_scores_table)

List of adjectives that pitbulls most outperform other dogs in
            breed adj most align  ...  adj least align cosine least align
0   affenpinscher        Hideous  ...       productive          -0.153087
1           akita       deformed  ...       meaningful          -0.148529
2      australian          dodgy  ...          cordial          -0.082133
3         basenji       sociable  ...       meaningful          -0.085070
4          beagle         widdle  ...         Negative          -0.061651
..            ...            ...  ...              ...                ...
59        terrier        vicious  ...            ample          -0.065694
60         vizsla         widdle  ...        equitable          -0.086334
61     weimaraner       sociable  ...       meaningful          -0.094788
62        whippet         classy  ...         damaging          -0.083452
63      wolfhound         widdle  ...        important          -0.084980

[64 rows x 5 columns]


# Experimentation

In [ ]:
# def load_word2vec():
#     """ Load Word2Vec Vectors
#         Return:
#             wv_from_bin: All 3 million embeddings, each lengh 300
#     """
#     import gensim.downloader as api
#     # wv_from_bin = api.load("word2vec-google-news-300")
#     wv_from_bin = api.load("glove-wiki-gigaword-200")
#     vocab = list(wv_from_bin.vocab.keys())
#     print("Loaded vocab size %i" % len(vocab))
#     return wv_from_bin
  
# # -----------------------------------
# # Run Cell to Load Word Vectors
# # Note: This may take several minutes
# # -----------------------------------
# wv_from_bin = load_word2vec()
# type(wv_from_bin)

In [ ]:
# from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import gensim
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 
from mpl_toolkits.mplot3d import axes3d
import matplotlib.pyplot as plt
from numpy.random import rand
from IPython.display import HTML
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from mpl_toolkits.mplot3d.proj3d import proj_transform
import matplotlib.animation as manimation
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
from gensim.models import KeyedVectors
import math
manimation.writers.list()

In [ ]:
model = wv_from_bin
emb = [model[word] for word in model.vocab]
words = list(model.wv.vocab)

# using PCA for compression
pca = PCA(n_components = 3)
emb_3d = pca.fit_transform(emb)
# model.vocab

In [ ]:
# making the blue arrow

class Arrow3D(FancyArrowPatch):

    def __init__(self, xs, ys, zs, *args, **kwargs):
      # xs, ys, zs is the x pair of initial and final points of vector I'm guessing
        FancyArrowPatch.__init__(self, (0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, renderer.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        FancyArrowPatch.draw(self, renderer)

In [ ]:
emb_3d

In [ ]:
def animate(frame, fig, ax, all_x_coors, all_y_coors, all_z_coors):
    # x, y, z  = emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2]
    slowness = 50
    swing_amount = 35
    # Use sin to simulate a natrual swing oscillation, but also want only positive numbers so add 1 (lowest sin is -1)
    ax.view_init(10, (math.sin(frame/slowness) + 1) * swing_amount)
    #plt.pause(.001)
    ax.set_xlim(min(all_x_coors), max(all_x_coors))
    ax.set_ylim(min(all_y_coors), max(all_y_coors))
    ax.set_zlim(min(all_z_coors), max(all_z_coors))
    return fig
 
def plot(fig, ax, embedding_start_vector, embedding_end_vector, labels):
  # Note each of these lists contain n elements, one for each vector coordinate
  # fig.clear(False)
  # fig = plt.figure(dpi=100)
  # ax = fig.gca(projection='3d')
  all_x_coors = [ele[0] for ele in (embedding_start_vector + embedding_end_vector)]
  all_y_coors = [ele[1] for ele in (embedding_start_vector + embedding_end_vector)]
  all_z_coors = [ele[2] for ele in (embedding_start_vector + embedding_end_vector)]
  current_title = ""
  color = ""
  not_initiated = True
  for start_vector, end_vector, label in zip(reversed(embedding_start_vector), reversed(embedding_end_vector), reversed(labels)):
    # If resultant vector, label shouldn't come at the end and interfere
    if not_initiated:
      current_title = label
      color = "y"
      not_initiated = False
    if not not_initiated and label != current_title:
      color = "b"
    if start_vector[0] != 0:
      ax.text((start_vector[0] + end_vector[0])/2,  (start_vector[1] + end_vector[1])/2,
             (start_vector[2] + end_vector[2])/2, '%s'%(str(label)),
            size = 13, zorder=1)  
    else:
      ax.text(end_vector[0], end_vector[1],
              end_vector[2], '%s'%(str(label)),
              size = 13, zorder=1)
    a = Arrow3D([start_vector[0], end_vector[0]], [start_vector[1], end_vector[1]],
                [start_vector[2], end_vector[2]], mutation_scale=20,
                lw=1, arrowstyle="-|>", color=color)
    ax.add_artist(a)   
  plt.plot([0],[0],[0], 'ro')
  anim = animation.FuncAnimation(fig, animate, frames=200, fargs=(fig,ax, all_x_coors, all_y_coors, all_z_coors), interval=50)
  return HTML(anim.to_html5_video())

In [ ]:
def add_vector(start_vector, end_vector, label, color, adj_start_vectors, adj_end_vectors, labels):
  adj_start_vectors.append(start_vector)
  adj_end_vectors.append(end_vector)
  labels.append(label)

# Get list of all end_vectors
adj = ['great', 'aggressive', 'green', 'japanese', 'good', 'bad']
# adj = ['good', 'bad']
adj_end_vectors = [emb for emb, word in zip(emb_3d, words) if word in adj]
labels = adj
adj_start_vectors = [np.array([0,0,0])] * len(labels)
colors = ["b"] * len(labels)
# Added goodness vector to start, end vector points and label
good_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'good'][0]
bad_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'bad'][0]
goodness_vector_coors = [x1 - x2 for x1, x2 in zip(good_coors, bad_coors)]
# Note Run below instead of two below to use the actual goodness_vector pre-added to vocab from OG word-vector and PCA'ed instead of pseudo 
# adding and subtracting good and bad PCA'ed vectors which is essentially not really mapping the goodness unit vector but rather just the resultant good-bad.
# Currently do good-bad for illustration purposes because should be technically equal if not for PCA'ed on 3D
# add_vector(bad_coors, [x1 + x2 for x1, x2 in zip(goodness_vector_coors, bad_coors)], "goodness_vector", "y", adj_start_vectors, adj_end_vectors, labels, colors)

In [ ]:
fig = plt.figure(dpi=100)
ax = fig.gca(projection='3d')
# plot(fig, ax, adj_start_vectors, adj_end_vectors, labels, all_x_coors, all_y_coors, all_z_coors)
add_vector(bad_coors, [x1 + x2 for x1, x2 in zip(goodness_vector_coors, bad_coors)], "goodness_vector", "y", adj_start_vectors, adj_end_vectors, labels)
plot(fig, ax, adj_start_vectors, adj_end_vectors, labels)

Demo Ready Implementation

In [ ]:
# This complicated gives new titles, frames animated adding
def animate(frame, fig, ax, embedding_start_vector, embedding_end_vector, labels, frame_time_stamp_ranges, axis_titles):
    # x, y, z  = emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2]
    # ax = fig.gca(projection='3d')
    all_x_coors = [ele[0] for ele in (adj_start_vectors + adj_end_vectors)]
    all_y_coors = [ele[1] for ele in (adj_start_vectors + adj_end_vectors)]
    all_z_coors = [ele[2] for ele in (adj_start_vectors + adj_end_vectors)]
    slowness = 50
    swing_amount = 35
    ax.view_init(10, (math.sin(frame/slowness) + 1) * swing_amount)
    current_title = ""
    not_initiated = True
    color = "y"
    for start_vector, end_vector, label, frame_time_stamp_ranges_for_vec, axis_title in zip(reversed(embedding_start_vector), reversed(embedding_end_vector), reversed(labels), reversed(frame_time_stamp_ranges), reversed(axis_titles)):
    # If resultant vector, label shouldn't come at the end and interfere
      alpha = 0
      for (min_frame_time_stamp, max_frame_time_stamp) in frame_time_stamp_ranges_for_vec:
        if frame >= min_frame_time_stamp and frame < max_frame_time_stamp:
          alpha = 1
      if alpha == 1 and not_initiated:
          current_title = axis_title
          color = "y"
          not_initiated = False
          ax.set_title(axis_title)
      if not not_initiated and axis_title != current_title:
        color = "b"
      if start_vector[0] != 0:
        ax.text((start_vector[0] + end_vector[0])/2,  (start_vector[1] + end_vector[1])/2,
              (start_vector[2] + end_vector[2])/2, '%s'%(str(label)),
              size = 13, zorder=1)  
      else:
        ax.text(end_vector[0], end_vector[1],
                end_vector[2], '%s'%(str(label)),
                size = 13, zorder=1)
      # if alpha==0:
        # print("invisible line")
      # else:
        # print("visible lin")
      a = Arrow3D([start_vector[0], end_vector[0]], [start_vector[1], end_vector[1]],
                  [start_vector[2], end_vector[2]], mutation_scale=20,
                  lw=1, arrowstyle="-|>", color=color, alpha=alpha)
      ax.add_artist(a)
    plt.plot([0],[0],[0], 'ro')
    #plt.pause(.001)
    ax.set_xlim(min(all_x_coors), max(all_x_coors))
    ax.set_ylim(min(all_y_coors), max(all_y_coors))
    ax.set_zlim(min(all_z_coors), max(all_z_coors))
    return fig
 
def plot(fig,ax, total_frames, embedding_start_vector, embedding_end_vector, labels, frame_time_stamp_ranges, axis_titles):
  # Note each of these lists contain n elements, one for each vector coordinate
  anim = animation.FuncAnimation(fig, animate, frames=total_frames, fargs=(fig,ax, embedding_start_vector, embedding_end_vector, labels, frame_time_stamp_ranges, axis_titles), interval=50)
  return HTML(anim.to_html5_video())

In [ ]:
def add_vector(start_vector, end_vector, label, color, adj_start_vectors, adj_end_vectors, labels):
  adj_start_vectors.append(start_vector)
  adj_end_vectors.append(end_vector)
  labels.append(label)
  
fig = plt.figure(dpi=100)
ax = fig.gca(projection='3d')

# Get list of all end_vectors
adj = ['great', 'aggressive', 'green', 'japanese', 'good', 'bad']
# adj = ['good', 'bad']
adj_end_vectors = [emb for emb, word in zip(emb_3d, words) if word in adj]
labels = adj
adj_start_vectors = [np.array([0,0,0])] * len(labels)
colors = ["b"] * len(labels)
# Added goodness vector to start, end vector points and label
good_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'good'][0]
bad_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'bad'][0]
goodness_vector_coors = [x1 - x2 for x1, x2 in zip(good_coors, bad_coors)]
add_vector(bad_coors, [x1 + x2 for x1, x2 in zip(goodness_vector_coors, bad_coors)], "goodness_vector", "y", adj_start_vectors, adj_end_vectors, labels)

# plot(fig, ax, 300, adj_start_vectors, adj_end_vectors, labels, [0,0,0,0, 100, 100, 200], ["Initial Vectors"]*4 + ["Good and Bad Vector Added"]*2 + ["Goodness Component Added vec(good) - vec(bad)"] )
plot(fig, ax, 330, adj_start_vectors, adj_end_vectors, labels, [[(0,100), (100,110), (120,130)], [(0,100), (10,11), (12,13)], [(0,10), (10,11), (12,13)], [(0,10), (10,11), (12,13)], [(10,20), (20,21), (22,23)],[(10,20), (20,21), (22,23)], [(20,30), (30,31), (32,33)]], ["Initial Vectors"]*4 + ["Good and Bad Vector Added"]*2 + ["Goodness Component Added vec(good) - vec(bad)"] )


In [ ]:
fig = plt.figure(dpi=100)
ax = fig.gca(projection='3d')

# Get list of all end_vectors
adj = []
# adj = ['good', 'bad']
adj_end_vectors = [emb for emb, word in zip(emb_3d, words) if word in adj]
labels = adj
adj_start_vectors = [np.array([0,0,0])] * len(labels)
colors = ["b"] * len(labels)
# Added goodness vector to start, end vector points and label
good_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'good'][0]
bad_coors = [vec for vec, word in zip(adj_end_vectors, labels) if word == 'bad'][0]
goodness_vector_coors = [x1 - x2 for x1, x2 in zip(good_coors, bad_coors)]
goodness_unit_vector = goodness_vector_coors / np.linalg.norm(goodness_vector_coors)
add_vector(bad_coors, [x1 + x2 for x1, x2 in zip(goodness_unit_vector, bad_coors)], "goodness_vector", "y", adj_start_vectors, adj_end_vectors, labels)

# plot(fig, ax, 300, adj_start_vectors, adj_end_vectors, labels, [0,0,0,0, 100, 100, 200], ["Initial Vectors"]*4 + ["Good and Bad Vector Added"]*2 + ["Goodness Component Added vec(good) - vec(bad)"] )
plot(fig, ax, 330, adj_start_vectors, adj_end_vectors, labels, [[(0,100), (100,110), (120,130)], [(0,100), (10,11), (12,13)], [(0,10), (10,11), (12,13)], [(0,10), (10,11), (12,13)], [(10,20), (20,21), (22,23)],[(10,20), (20,21), (22,23)], [(20,30), (30,31), (32,33)]], ["Initial Vectors"]*4 + ["Good and Bad Vector Added"]*2 + ["Goodness Component Added vec(good) - vec(bad)"] )


# Algo:

* list of adjectives
* Get list of word vectors for the adjectives
* Separately find/define a "goodness unit vector"
* Find the cosine between each adjective and the goodness vector 
* Sort the list, get the best and the worst words


Input: [happy, japanese, green, worse, smile]


Output:

Good words: [happy, smile]

Bad words: [worse]